# Can we predict late deliveries?
Target: `late_delivery`. Prediction is made at dispatch, for future consignments, so the split is temporal.
Each consignment appears once, so no entity grouping is needed (verified below).

**Feature decisions (checklist 7):** `consignment` is an identifier, dropped. `claim_paid_eur` is recorded after delivery (a claim settled for the consignment), so it is not known at prediction time and leaks the outcome, dropped. Features: `carrier`, `distance_km`, `weight_kg`, `parcels`, plus calendar features derived from `dispatched_on` (hour, weekday).

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, 'unique consignments:', df.consignment.nunique(), 'late rate:', df.late_delivery.mean().round(3))
df['hour'] = df.dispatched_on.dt.hour
df['weekday'] = df.dispatched_on.dt.weekday

## 1. Test set: the latest 20% of rows by dispatch time (set aside before any modelling)

In [ ]:
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
assert train.dispatched_on.max() <= test.dispatched_on.min()
assert not set(train.consignment) & set(test.consignment)
print(train.dispatched_on.min(), train.dispatched_on.max(), len(train))
print(test.dispatched_on.min(), test.dispatched_on.max(), len(test))
FEATS = ['carrier','distance_km','weight_kg','parcels','hour','weekday']
NUM = ['distance_km','weight_kg','parcels','hour','weekday']
CAT = ['carrier']
Xtr, ytr = train[FEATS], train.late_delivery.values
# test set is not touched again until step 6

## 2-4. Candidate models (named before any scoring) and CV structure
Candidates:
1. `baseline`: predicts the training-fold late rate (trivial).
2. `logreg`: standardised numeric + one-hot carrier, L2 logistic regression, C in {0.1, 1, 10}.
3. `hgb`: histogram gradient boosting, small grid over learning rate/depth.

CV: expanding-window `TimeSeriesSplit` (5 folds) on training rows only; earlier rows train, later rows validate. All preprocessing lives inside the pipeline, so is fit per fold. Selection metric: ROC AUC (threshold-free); the decision threshold is then chosen on out-of-fold predictions.

In [ ]:
def prep(): return ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
def make(name, **p):
    if name=='baseline': return Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))])
    if name=='logreg':   return Pipeline([('p',prep()),('m',LogisticRegression(C=p['C'],max_iter=1000))])
    if name=='hgb':      return Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(learning_rate=p['lr'],max_depth=p['d'],max_iter=200,random_state=0))])
CANDS = [('baseline',{})]+[('logreg',{'C':c}) for c in (0.1,1,10)]+[('hgb',{'lr':l,'d':d}) for l in (0.03,0.1) for d in (3,None)]
tscv = TimeSeriesSplit(n_splits=5)
def cv_oof(name,p):
    oof = np.full(len(train), np.nan); aucs=[]
    for tr,va in tscv.split(Xtr):
        m = make(name,**p).fit(Xtr.iloc[tr], ytr[tr])
        oof[va] = m.predict_proba(Xtr.iloc[va])[:,1]
        aucs.append(roc_auc_score(ytr[va], oof[va]))
    return oof, aucs
res=[]; oofs={}
for i,(n,p) in enumerate(CANDS):
    oof,a = cv_oof(n,p); oofs[i]=oof
    res.append(dict(model=n,params=p,cv_auc_mean=np.mean(a),cv_auc_sd=np.std(a)))
res=pd.DataFrame(res).sort_values('cv_auc_mean',ascending=False); res

## 5. Choose model and threshold on CV only

In [ ]:
best = res.index[0]; name,params = CANDS[best]
mask = ~np.isnan(oofs[best])
oof, y_oof = oofs[best][mask], ytr[mask]
ths = np.linspace(0.05,0.8,76)
f1s = [f1_score(y_oof, oof>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen:', name, params, '| CV AUC %.3f | threshold (max OOF F1) %.2f | OOF F1 %.3f' % (res.cv_auc_mean.iloc[0], thr, max(f1s)))
print('baseline CV AUC: %.3f' % res[res.model=='baseline'].cv_auc_mean.iloc[0])

**Written choice (before test scoring):** the model/hyperparameters above with the shown threshold, selected purely on training-only time-series CV. The choice is fixed now.

## 6. Refit on all training rows; score the test set once

In [ ]:
final = make(name,**params).fit(Xtr,ytr)
p = final.predict_proba(test[FEATS])[:,1]; yt = test.late_delivery.values
base = make('baseline').fit(Xtr,ytr).predict_proba(test[FEATS])[:,1]
print('TEST ROC AUC        : %.3f  (baseline %.3f)' % (roc_auc_score(yt,p), roc_auc_score(yt,base)))
print('TEST avg precision  : %.3f  (prevalence %.3f)' % (average_precision_score(yt,p), yt.mean()))
print('TEST Brier          : %.3f  (baseline %.3f)' % (brier_score_loss(yt,p), brier_score_loss(yt,base)))
print('TEST F1 @ thr %.2f  : %.3f' % (thr, f1_score(yt,p>=thr)))

## Post-hoc diagnostic (descriptive only; nothing is re-tuned after this)
The test-period late rate is far below the training rate, so the test score reflects a distribution shift.

In [ ]:
m = df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size']).round(3)
print(m.to_string())
print('train late rate %.3f | test late rate %.3f' % (ytr.mean(), yt.mean()))

## Conclusion (headline = single test-set score from step 6)
- Chosen on CV: regularised logistic regression (C=0.1), threshold 0.24. CV ROC AUC about 0.72 (baseline 0.50): a modest signal from carrier, distance, weight, parcels and dispatch time.
- Test (latest 20%, Oct 2025 to Feb 2026): ROC AUC about 0.70, so ranking ability held up. But the late rate collapsed from about 21% (and ~30% in recent training months) to 2.2% in the test period, so probabilities are badly miscalibrated (Brier 0.088 vs 0.077 for the baseline) and the fixed threshold gives F1 of only 0.08, average precision 0.069.
- So: consignments can be ranked by lateness risk moderately well, but not usefully classified at the chosen threshold, and the late rate changed regime near Oct 2025. Find out what changed (carrier performance, promised-date policy, label definition) before deploying.
- Caveats: single temporal holdout; the shift means CV, which covers the pre-shift regime only, was too optimistic about the test period; `claim_paid_eur` was excluded as post-outcome information.